# 第4章　`nn.Module` でニューラルネット & 分類

直線だけでは曲がった境界を学べません。**層を重ね、間に活性化関数（非線形）を挟む**とニューラルネットになります。
この章では `nn.Module` でモデルを組み立て、**分類問題**（クラス当て）を解きます。

この章のゴール：自分のモデルクラスを定義でき、分類の損失 `CrossEntropyLoss` を正しく使える。

> **このノートの使い方**
> - 上から順にセルを実行（Colab/Jupyter ともに `Shift + Enter`）。
> - コードは**少し書き換えて壊して直す**のが一番伸びます。各章末に演習があります。
> - GPU は不要な章が多いです。重い章（CNN）では使い方を案内します。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 4-1. 部品：`nn.Linear` / 活性化 / `nn.Sequential`

- `nn.Linear(in, out)`：全結合層（前の章の直線の多次元版）。
- 活性化関数 `nn.ReLU()` など：**非線形性**を与える。これが無いと何層重ねても直線1枚と同じ。
- `nn.Sequential(...)`：層を順番に並べる手軽な書き方。

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(2, 16),   # 入力2次元 -> 16
    nn.ReLU(),          # 非線形
    nn.Linear(16, 2),   # -> 出力2（2クラス分のスコア）
)
print(model)

x = torch.randn(5, 2)        # 5サンプル, 各2次元
print("出力 shape:", model(x).shape)   # (5, 2)

## 4-2. `nn.Module` をクラスで定義（実務の基本形）

複雑なモデルは `nn.Module` を継承したクラスで書きます。約束は2つだけ：
- `__init__` で**使う層を作る**（`super().__init__()` を忘れず）。
- `forward` で**データの流れ**を書く。

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden=16, out_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.fc1(x))   # 1層目 -> ReLU
        x = self.fc2(x)             # 2層目（出力。ここでは活性化しない）
        return x

model = MLP()
print(model)

## 4-3. 分類データを作る（2つの集団）
2次元平面に、左下の集団（クラス0）と右上の集団（クラス1）を作ります。

In [ ]:
torch.manual_seed(0)
n = 200
c0 = torch.randn(n, 2) + torch.tensor([-2.0, -2.0])   # クラス0
c1 = torch.randn(n, 2) + torch.tensor([ 2.0,  2.0])   # クラス1
X = torch.cat([c0, c1], dim=0)                        # (400, 2)
yv = torch.cat([torch.zeros(n), torch.ones(n)]).long()  # ラベルは整数 (400,)
print("X:", X.shape, " y:", yv.shape, " yの種類:", yv.unique())

## 4-4. 分類の損失：`CrossEntropyLoss`（つまずき注意）

分類では `nn.CrossEntropyLoss` を使います。**ここが初心者の最大の罠**：

- モデルの出力は **生のスコア（logits）** のまま渡す。**自分で `softmax` をかけない**（損失関数が内部でやる）。
- 正解ラベルは **クラス番号の整数（long）**。one-hot にしない。

> softmax はスコアを「確率（合計1）」に変換する関数。予測クラスを見たいときだけ最後に使う。

In [ ]:
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(100):
    optimizer.zero_grad()
    logits = model(X)            # 生スコア (400, 2)。softmaxしない！
    loss = criterion(logits, yv) # 正解は整数ラベル
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}: loss={loss.item():.4f}")

## 4-5. 正解率（accuracy）を測る ＆ 評価のお作法

評価のときは：
- `model.eval()`（評価モードに切替。Dropout等の挙動が変わる）
- `with torch.no_grad():`（勾配不要 → 速い・省メモリ）
- 予測クラスは logits の **最大の位置** = `argmax(dim=1)`。

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X)
    probs = torch.softmax(logits, dim=1)   # 確率にしたいときはここで
    pred = logits.argmax(dim=1)            # 予測クラス（0 or 1）
    acc = (pred == yv).float().mean()

print("最初の3件の確率:\n", probs[:3])
print("正解率 accuracy =", acc.item())

## 4-6. 決定境界を可視化（おまけ）
モデルが平面をどう2色に塗り分けたかを見ます。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xx, yy = torch.meshgrid(torch.linspace(-6, 6, 200),
                        torch.linspace(-6, 6, 200), indexing="xy")
grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
with torch.no_grad():
    zz = model(grid).argmax(dim=1).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx.numpy(), yy.numpy(), zz.numpy(), alpha=0.3)
plt.scatter(X[:, 0], X[:, 1], c=yv, s=10, cmap="bwr")
plt.title("Decision boundary"); plt.show()

## 演習 4
1. クラスを**3つ**に増やそう（3つ目の集団を追加し、出力を `nn.Linear(16, 3)` に。ラベルは 0/1/2）。`CrossEntropyLoss` はそのまま使える。
2. 隠れ層の幅 `hidden` を 4 / 64 に変えて境界の滑らかさを比較しよう。
3. 活性化 `ReLU` を消す（`forward` で `self.fc1(x)` 直結に）と境界が直線になることを確認しよう＝非線形の効果。

In [ ]:
# ここに自分のコードを書いて実行してみよう
